# 16 - ConvLSTM Optical Flow and Grad-CAM Comparison

This notebook compares EF Grad-CAM saliency with optical flow for the motion-head multitask bidirectional ConvLSTM U-Net. It generates `ef_primary_motion` EF Grad-CAM directly for both `encoder_bottleneck` and `temporal_representation`, computes RAFT-Large optical flow on the exact 23 input frames used by the model, and writes transition-level spatial/temporal comparison outputs.


## Setup

Kaggle paths are configurable through environment variables. Keep `RUN_MODE = "smoke"` for a quick validation run; switch to `"full"` to process the official test set. This notebook selects representative visualization samples itself and does not require precomputed Grad-CAM NPZ files from notebook 14.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import sys
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

try:
    import torchvision
    from torchvision.models.optical_flow import Raft_Large_Weights, raft_large
except Exception as exc:
    raise ImportError("TorchVision with RAFT optical-flow models is required for this notebook.") from exc


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/input/src-updated",
    "/kaggle/input/datasets/jiyoonoh24/echonet-src-code",
    "/kaggle/input/datasets/sooahnoh/echonet-code-updated-4",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.bidirectional_convlstm_unet import build_bidirectional_convlstm_unet
from src.gradcam_ef_regression import (
    EchoNetTemporalEFDataset,
    build_ef_regression_convlstm,
    denormalize_ef,
    encoder_bottleneck_ef_gradcam,
    load_exact_checkpoint,
    regression_metrics,
    save_gradcam_npz,
    select_representative_samples,
    temporal_representation_ef_probe_gradcam,
)
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"
TRAINED_RUN_DIR = Path(os.environ.get(
    "EF_MOTION_CONVLSTM_RUN_DIR",
    "/kaggle/input/ef-primary-motion-head-conv-lstm-07-22" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "ef_primary_motion_head_conv_lstm_07_22",
))
SEGMENTATION_RUN_DIR = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_SEG_RUN_DIR",
    "/kaggle/input/bidirectional-convlstm-unet-23-frames" if Path("/kaggle/input").exists() else PROJECT_ROOT / "outputs" / "runs" / "bidirectional_convlstm_unet_23_frames",
))
RUN_DIR = Path(os.environ.get(
    "FLOW_GRADCAM_COMPARISON_RUN_DIR",
    "/kaggle/working/outputs/runs/convlstm_optical_flow_gradcam_comparison" if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "convlstm_optical_flow_gradcam_comparison",
))
GRADCAM_MODEL_DIR = RUN_DIR / "generated_gradcam" / "ef_primary_motion"
MANIFEST_DIR = RUN_DIR / "manifests"
INFERENCE_DIR = RUN_DIR / "inference_npz"
FLOW_DIR = RUN_DIR / "flow_npz"
FLOW_ANALYSIS_DIR = RUN_DIR / "flow_analysis_npz"
FIGURE_DIR = RUN_DIR / "figures"
VIDEO_DIR = RUN_DIR / "videos"
for directory in [RUN_DIR, MANIFEST_DIR, INFERENCE_DIR, FLOW_DIR, FLOW_ANALYSIS_DIR, FIGURE_DIR, VIDEO_DIR, GRADCAM_MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Trained motion-head run: {TRAINED_RUN_DIR}")
print(f"Segmentation ConvLSTM run: {SEGMENTATION_RUN_DIR}")
print(f"Generated Grad-CAM model directory: {GRADCAM_MODEL_DIR}")
print(f"Comparison output directory: {RUN_DIR}")


## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for the official test set

CONFIG = {
    "seed": 42,
    "num_frames_before": 11,
    "num_frames_after": 11,
    "temporal_stride": 2,
    "target_idx": 11,
    "sequence_length": 23,
    "image_size": [112, 112],
    "channels": [16, 32, 64, 128],
    "ef_hidden_dim": 128,
    "dropout": 0.1,
    "motion_hidden_channels": 64,
    "segmentation_threshold": 0.5,
    "batch_size_for_inference": 4,
    "dynamic_segmentation_batch_size": 4,
    "num_workers": 2,
    "smoke_max_samples": 12,
    "visualization_sample_count": 10,
    "full_npz_sample_count": 20,
    "save_full_npz_for_all_samples": False,
    "flow_array_storage_dtype": "float16",
    "raft_transition_batch_size": 8,
    "raft_min_input_size": 128,
    "raft_pad_mode": "replicate",
    "flow_valid_error_threshold_px": 3.0,
    "boundary_width_px": 3,
    "gradcam_layers": ["encoder_bottleneck", "temporal_representation"],
    "gradcam_analysis_cam_key": "positive_cams",
    "gradcam_visual_cam_key": "frame_normalized_positive_cams",
    "topk_fraction": 0.10,
    "video_fps": 5,
}

run_config_path = TRAINED_RUN_DIR / "config.json"
if run_config_path.exists():
    trained_config = json.loads(run_config_path.read_text())
    for key in [
        "num_frames_before", "num_frames_after", "temporal_stride", "target_idx", "sequence_length",
        "image_size", "channels", "ef_hidden_dim", "dropout", "motion_hidden_channels",
        "segmentation_threshold",
    ]:
        if key in trained_config:
            CONFIG[key] = trained_config[key]

CONFIG["run_mode"] = RUN_MODE
set_seed(int(CONFIG["seed"]))
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
CONFIG


## PART 1 - Model Inference Dataset

This reconstructs the official EchoNet-Dynamic test split using the same target-centered 23-frame sampling used in notebook 14. The dataset sequence tensor is the pre-normalized grayscale input in `[0,1]`, so it is saved directly for optical-flow alignment.


In [ ]:

metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"
GRADCAM_MODEL_DIR.mkdir(parents=True, exist_ok=True)

samples = load_temporal_metadata(metadata_path)
file_list, _volume_tracings = load_echonet_tables(RAW_DIR)
assert "EF" in file_list.columns, "FileList.csv must contain EF labels."

echo_table = file_list.copy()
echo_table["video_stem"] = echo_table["FileName"].astype(str).map(lambda x: Path(x).stem)
ef_lookup = dict(zip(echo_table["video_stem"], echo_table["EF"].astype(float)))

samples_with_ef = []
for sample in samples:
    item = dict(sample)
    video_stem = Path(str(item["video_id"])).stem
    if video_stem in ef_lookup and pd.notna(ef_lookup[video_stem]):
        item["ef"] = float(ef_lookup[video_stem])
        samples_with_ef.append(item)

gt_mask_lookup = {}
for sample in samples_with_ef:
    key = (str(sample["video_id"]), int(sample["frame_idx"]))
    gt_mask_lookup[key] = str(sample["mask"])

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples_with_ef, file_list)
ef_values_train = np.array([float(sample["ef"]) for sample in train_samples], dtype=np.float32)
ef_mean = float(ef_values_train.mean())
ef_std = float(ef_values_train.std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero."

base_test_dataset = EchoNetTemporalDataset(
    test_samples,
    videos_dir=VIDEOS_DIR,
    num_frames_before=int(CONFIG["num_frames_before"]),
    num_frames_after=int(CONFIG["num_frames_after"]),
    temporal_stride=int(CONFIG["temporal_stride"]),
    image_size=tuple(CONFIG["image_size"]),
    augment=False,
)
test_dataset = EchoNetTemporalEFDataset(base_test_dataset, ef_mean=ef_mean, ef_std=ef_std)
sample_id_to_full_index = {str(sample["id"]): idx for idx, sample in enumerate(test_dataset.base_dataset.samples)}

if RUN_MODE == "smoke":
    active_indices = list(range(min(int(CONFIG["smoke_max_samples"]), len(test_dataset))))
else:
    active_indices = list(range(len(test_dataset)))

active_sample_ids = [str(test_dataset.base_dataset.samples[idx]["id"]) for idx in active_indices]
visualization_sample_ids = active_sample_ids[: int(CONFIG["visualization_sample_count"])]
full_npz_sample_ids = active_sample_ids[: int(CONFIG["full_npz_sample_count"])]
full_npz_sample_id_set = set(full_npz_sample_ids)
visualization_sample_id_set = set(visualization_sample_ids)
storage_plan_df = pd.DataFrame({
    "sample_id": active_sample_ids,
    "save_full_npz_arrays": [sid in full_npz_sample_id_set or bool(CONFIG["save_full_npz_for_all_samples"]) for sid in active_sample_ids],
    "save_visualizations": [sid in visualization_sample_id_set for sid in active_sample_ids],
    "selection_phase": "provisional_before_inference",
})
storage_plan_df.to_csv(MANIFEST_DIR / "sample_storage_plan.csv", index=False)

active_dataset = Subset(test_dataset, active_indices)
inference_loader = DataLoader(
    active_dataset,
    batch_size=int(CONFIG["batch_size_for_inference"]),
    shuffle=False,
    num_workers=int(CONFIG["num_workers"]),
    pin_memory=torch.cuda.is_available(),
)
print({
    "train": len(train_samples),
    "validation": len(val_samples),
    "test": len(test_samples),
    "active": len(active_dataset),
    "run_mode": RUN_MODE,
    "ef_mean": ef_mean,
    "ef_std": ef_std,
})


## PART 1 - Load Motion-Head Model


In [ ]:
checkpoint_path = Path(os.environ.get(
    "EF_PRIMARY_MOTION_CHECKPOINT_PATH",
    TRAINED_RUN_DIR / "checkpoints" / "ef_primary_motion" / "best_ef_mae.pt",
))
if not checkpoint_path.exists():
    candidates = sorted(TRAINED_RUN_DIR.rglob("*ef_primary_motion*best*ef*.pt")) + sorted(TRAINED_RUN_DIR.rglob("best_ef_mae.pt"))
    assert candidates, f"Missing motion-head checkpoint under {TRAINED_RUN_DIR}. Set EF_PRIMARY_MOTION_CHECKPOINT_PATH."
    checkpoint_path = candidates[0]

model = build_ef_regression_convlstm(CONFIG, with_motion=True).to(device)
checkpoint_metadata = load_exact_checkpoint(model, checkpoint_path, device="cpu")
model.to(device)
model.eval()
print(json.dumps({k: v for k, v in checkpoint_metadata.items() if k not in {"config", "metrics"}}, indent=2, default=str))
assert hasattr(model, "motion_head"), "This notebook must use the EF-primary model WITH the motion head."
seg_config_path = SEGMENTATION_RUN_DIR / "config.json"
seg_config = json.loads(seg_config_path.read_text()) if seg_config_path.exists() else {}
seg_checkpoint_path = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_SEG_CHECKPOINT_PATH",
    SEGMENTATION_RUN_DIR / "checkpoints" / "best_model.pt",
))
if not seg_checkpoint_path.exists():
    candidates = sorted((SEGMENTATION_RUN_DIR / "checkpoints").glob("*.pt")) if (SEGMENTATION_RUN_DIR / "checkpoints").exists() else []
    assert candidates, f"Missing segmentation checkpoint under {SEGMENTATION_RUN_DIR}. Set BIDIRECTIONAL_CONVLSTM_SEG_CHECKPOINT_PATH."
    seg_checkpoint_path = candidates[0]
segmentation_model = build_bidirectional_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(seg_config.get("channels", CONFIG["channels"])),
    num_frames_before=int(seg_config.get("num_frames_before", CONFIG["num_frames_before"])),
    num_frames_after=int(seg_config.get("num_frames_after", CONFIG["num_frames_after"])),
).to(device)
seg_checkpoint = torch.load(seg_checkpoint_path, map_location="cpu")
seg_state = seg_checkpoint.get("model_state_dict", seg_checkpoint.get("state_dict", seg_checkpoint))
if any(key.startswith("module.") for key in seg_state):
    seg_state = {key.removeprefix("module."): value for key, value in seg_state.items()}
segmentation_model.load_state_dict(seg_state, strict=True)
segmentation_model.eval()
segmentation_checkpoint_metadata = {
    "checkpoint_path": str(seg_checkpoint_path),
    "checkpoint_keys": sorted(list(seg_checkpoint.keys())) if isinstance(seg_checkpoint, dict) else [],
    "epoch": seg_checkpoint.get("epoch") if isinstance(seg_checkpoint, dict) else None,
    "metrics": seg_checkpoint.get("metrics") if isinstance(seg_checkpoint, dict) else None,
    "config": seg_config,
}
assert segmentation_model.expected_sequence_length == int(CONFIG["sequence_length"]), segmentation_model.expected_sequence_length
print("segmentation_model", json.dumps({k: v for k, v in segmentation_checkpoint_metadata.items() if k not in {"config", "metrics"}}, indent=2, default=str))

with (MANIFEST_DIR / "checkpoint_metadata.json").open("w", encoding="utf-8") as f:
    json.dump({"ef_primary_motion": checkpoint_metadata, "segmentation_dynamic_roi": segmentation_checkpoint_metadata}, f, indent=2, default=str)


## PART 1 - Save Exact Inputs and Model Outputs

For LV-restricted optical-flow summaries, this notebook now generates dynamic per-frame LV ROIs. Each sampled frame is segmented by shifting a 23-frame ConvLSTM input so that the frame being segmented is the center target frame. When a ground-truth ED/ES mask is available for a sampled frame, that ground-truth mask replaces the predicted mask for ROI generation. Transition ROIs are then built from the union of masks at t and t+1.


In [ ]:
def upsample_motion_head(flow_pred: torch.Tensor, output_hw: tuple[int, int]) -> torch.Tensor:
    """Upsample model motion-head output [B,T-1,2,h,w] to [B,T-1,2,H,W] and scale uv pixels."""
    b, transitions, channels, h, w = flow_pred.shape
    H, W = output_hw
    flat = flow_pred.reshape(b * transitions, channels, h, w)
    up = F.interpolate(flat, size=output_hw, mode="bilinear", align_corners=False)
    up[:, 0] *= float(W) / float(w)
    up[:, 1] *= float(H) / float(h)
    return up.reshape(b, transitions, channels, H, W)



def boundary_mask(mask: np.ndarray, width: int = 3) -> np.ndarray:
    mask_u8 = (mask > 0).astype(np.uint8)
    kernel = np.ones((max(1, width), max(1, width)), np.uint8)
    dilated = cv2.dilate(mask_u8, kernel, iterations=1)
    eroded = cv2.erode(mask_u8, kernel, iterations=1)
    return ((dilated - eroded) > 0)


def read_gt_mask(video_id: str, frame_idx: int) -> np.ndarray | None:
    path = gt_mask_lookup.get((str(video_id), int(frame_idx)))
    if path is None:
        return None
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    mask = cv2.resize(mask, (int(CONFIG["image_size"][1]), int(CONFIG["image_size"][0])), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.uint8)


@torch.inference_mode()
def generate_dynamic_frame_masks(video_id: str, sampled_frame_indices: np.ndarray) -> dict[str, np.ndarray]:
    """Segment each sampled frame as the center frame of its own 23-frame ConvLSTM context."""
    contexts = []
    frame_indices_per_context = []
    for frame_idx in sampled_frame_indices.astype(int).tolist():
        sequence, context_indices, _frame_count = base_test_dataset._read_sequence(base_test_dataset._video_path(str(video_id)), int(frame_idx))
        contexts.append(sequence[:, None].astype(np.float32))
        frame_indices_per_context.append(np.asarray(context_indices, dtype=np.int32))
    contexts_np = np.stack(contexts, axis=0)
    probs = []
    batch_size = int(CONFIG["dynamic_segmentation_batch_size"])
    for start in range(0, contexts_np.shape[0], batch_size):
        chunk = torch.from_numpy(contexts_np[start:start + batch_size]).to(device)
        logits = segmentation_model(chunk)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy()[:, 0].astype(np.float32))
    pred_prob = np.concatenate(probs, axis=0)
    pred_mask = (pred_prob >= float(CONFIG["segmentation_threshold"])).astype(np.uint8)
    roi_masks = pred_mask.copy()
    has_gt = np.zeros(len(sampled_frame_indices), dtype=np.uint8)
    roi_source = np.array(["predicted"] * len(sampled_frame_indices), dtype="U16")
    for i, frame_idx in enumerate(sampled_frame_indices.astype(int).tolist()):
        gt = read_gt_mask(str(video_id), int(frame_idx))
        if gt is not None:
            roi_masks[i] = gt.astype(np.uint8)
            has_gt[i] = 1
            roi_source[i] = "ground_truth"
    return {
        "dynamic_context_frame_indices": np.stack(frame_indices_per_context, axis=0).astype(np.int32),
        "dynamic_frame_seg_prob": pred_prob.astype(np.float32),
        "dynamic_frame_seg_mask_pred": pred_mask.astype(np.uint8),
        "dynamic_frame_roi_mask": roi_masks.astype(np.uint8),
        "dynamic_frame_has_ground_truth": has_gt,
        "dynamic_frame_roi_source": roi_source,
    }


def transition_rois_from_frame_masks(frame_roi_masks: np.ndarray, boundary_width_px: int) -> tuple[np.ndarray, np.ndarray]:
    cavity = []
    boundary = []
    boundaries = np.stack([boundary_mask(mask, width=boundary_width_px) for mask in frame_roi_masks], axis=0)
    for t in range(frame_roi_masks.shape[0] - 1):
        cavity.append(np.logical_or(frame_roi_masks[t] > 0, frame_roi_masks[t + 1] > 0))
        boundary.append(np.logical_or(boundaries[t] > 0, boundaries[t + 1] > 0))
    return np.stack(cavity, axis=0).astype(np.uint8), np.stack(boundary, axis=0).astype(np.uint8)


def save_inference_sample(batch: dict[str, Any], batch_index: int, out: dict[str, torch.Tensor], output_path: Path, save_full_npz: bool) -> dict[str, Any]:
    sequence = batch["sequence"][batch_index].detach().cpu().numpy().astype(np.float32)[:, 0]
    frame_indices = batch["frame_indices"][batch_index].detach().cpu().numpy().astype(np.int32)
    center_seg_prob = torch.sigmoid(out["seg_logits"][batch_index]).detach().cpu().numpy().astype(np.float32)[0]
    center_seg_mask = (center_seg_prob >= float(CONFIG["segmentation_threshold"])).astype(np.uint8)
    motion = upsample_motion_head(out["flow_pred"], tuple(CONFIG["image_size"]))[batch_index].detach().cpu().numpy().astype(np.float32)
    ef_pred = float(denormalize_ef(out["ef_normalized"][batch_index].detach().cpu(), ef_mean, ef_std))
    ef_true = float(batch["ef"][batch_index].detach().cpu())
    sample_id = str(batch["id"][batch_index])
    video_id = str(batch["video_id"][batch_index])
    target_idx = int(batch["target_idx"][batch_index])
    target_frame_idx = int(batch["frame_idx"][batch_index])
    timestamps = (frame_indices - frame_indices[target_idx]).astype(np.float32)
    dynamic = None
    transition_cavity_roi = None
    transition_boundary_roi = None
    gt_roi_frame_count = int(sum((str(video_id), int(frame_idx)) in gt_mask_lookup for frame_idx in frame_indices.tolist()))
    if save_full_npz:
        dynamic = generate_dynamic_frame_masks(video_id, frame_indices)
        transition_cavity_roi, transition_boundary_roi = transition_rois_from_frame_masks(
            dynamic["dynamic_frame_roi_mask"],
            boundary_width_px=int(CONFIG["boundary_width_px"]),
        )
        assert transition_cavity_roi.shape[0] == int(CONFIG["sequence_length"]) - 1
        gt_roi_frame_count = int(dynamic["dynamic_frame_has_ground_truth"].sum())
    metadata = {
        "sample_id": sample_id,
        "video_id": video_id,
        "target_idx": target_idx,
        "target_frame_idx": target_frame_idx,
        "frame_indices": frame_indices.tolist(),
        "timestamps_relative_frames": timestamps.tolist(),
        "temporal_stride": int(batch["temporal_stride"][batch_index]),
        "frame_count": int(batch["frame_count"][batch_index]),
        "ef_pred": ef_pred,
        "ef_true": ef_true,
        "segmentation_threshold": float(CONFIG["segmentation_threshold"]),
        "dynamic_roi_note": "Each sampled frame is segmented as the center frame of a shifted 23-frame ConvLSTM input. If a GT ED/ES mask is available for that frame, the GT mask replaces the predicted binary mask for ROI generation. Transition ROIs are unions of frame t and frame t+1 masks/boundaries.",
        "dynamic_segmentation_checkpoint": segmentation_checkpoint_metadata,
    }
    inference_npz_path = ""
    if save_full_npz:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            output_path,
            input_frames=sequence,
            sampled_frame_indices=frame_indices,
            timestamps_relative_frames=timestamps,
            target_idx=np.array(target_idx, dtype=np.int32),
            target_frame_idx=np.array(target_frame_idx, dtype=np.int32),
            ef_pred=np.array(ef_pred, dtype=np.float32),
            ef_true=np.array(ef_true, dtype=np.float32),
            seg_prob_target_from_multitask=center_seg_prob,
            seg_mask_target_from_multitask=center_seg_mask,
            dynamic_context_frame_indices=dynamic["dynamic_context_frame_indices"],
            dynamic_frame_seg_prob=dynamic["dynamic_frame_seg_prob"],
            dynamic_frame_seg_mask_pred=dynamic["dynamic_frame_seg_mask_pred"],
            dynamic_frame_roi_mask=dynamic["dynamic_frame_roi_mask"],
            dynamic_frame_has_ground_truth=dynamic["dynamic_frame_has_ground_truth"],
            dynamic_frame_roi_source=dynamic["dynamic_frame_roi_source"],
            transition_cavity_roi=transition_cavity_roi,
            transition_boundary_roi=transition_boundary_roi,
            motion_head_flow_uv=motion,
            sample_id=np.array(sample_id),
            video_id=np.array(video_id),
            metadata_json=np.array(json.dumps(metadata, default=str)),
        )
        inference_npz_path = str(output_path)
    return {
        "sample_id": sample_id,
        "video_id": video_id,
        "target_idx": target_idx,
        "target_frame_idx": target_frame_idx,
        "ef_pred": ef_pred,
        "ef_true": ef_true,
        "abs_ef_error": abs(ef_pred - ef_true),
        "sampled_frame_indices": "[" + ",".join(str(int(v)) for v in frame_indices.tolist()) + "]",
        "gt_roi_frame_count": gt_roi_frame_count,
        "save_full_npz_arrays": bool(save_full_npz),
        "inference_npz_path": inference_npz_path,
    }

inference_rows = []
with torch.inference_mode():
    for batch in tqdm(inference_loader, desc="model inference + dynamic LV masks"):
        sequence = batch["sequence"].to(device, non_blocking=True)
        out = model(sequence)
        assert "ef_normalized" in out and "seg_logits" in out and "flow_pred" in out
        assert out["flow_pred"].shape[1] == int(CONFIG["sequence_length"]) - 1, out["flow_pred"].shape
        for i in range(sequence.shape[0]):
            sample_id = str(batch["id"][i])
            save_full_npz = bool(CONFIG["save_full_npz_for_all_samples"]) or sample_id in full_npz_sample_id_set
            row = save_inference_sample(batch, i, out, INFERENCE_DIR / f"{sample_id}_inference_outputs.npz", save_full_npz=save_full_npz)
            inference_rows.append(row)

inference_df = pd.DataFrame(inference_rows)
inference_df.to_csv(MANIFEST_DIR / "inference_manifest.csv", index=False)
display(inference_df.head())


## Representative Samples from Existing Motion-Head Grad-CAM


In [ ]:

dataset_metrics = regression_metrics(inference_df["ef_pred"], inference_df["ef_true"])
with (MANIFEST_DIR / "normal_test_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(dataset_metrics, f, indent=2)

representative_sample_ids = select_representative_samples(
    inference_df,
    count=min(int(CONFIG["visualization_sample_count"]), len(inference_df)),
)
full_npz_sample_ids = select_representative_samples(
    inference_df,
    count=min(int(CONFIG["full_npz_sample_count"]), len(inference_df)),
)
for sample_id in representative_sample_ids:
    if sample_id not in full_npz_sample_ids:
        full_npz_sample_ids.append(sample_id)
full_npz_sample_ids = full_npz_sample_ids[: min(int(CONFIG["full_npz_sample_count"]), len(inference_df))]
for sample_id in representative_sample_ids:
    if sample_id not in full_npz_sample_ids:
        full_npz_sample_ids.append(sample_id)

full_npz_sample_id_set = set(full_npz_sample_ids)
visualization_sample_id_set = set(representative_sample_ids)

def materialize_full_inference_npz(sample_id: str) -> None:
    global inference_df
    row_idx = inference_df.index[inference_df["sample_id"] == sample_id]
    assert len(row_idx) == 1, sample_id
    row_idx = int(row_idx[0])
    existing = str(inference_df.loc[row_idx, "inference_npz_path"])
    if existing and Path(existing).exists():
        return
    full_index = sample_id_to_full_index[sample_id]
    loader = DataLoader(Subset(test_dataset, [full_index]), batch_size=1, shuffle=False, num_workers=0)
    batch = next(iter(loader))
    with torch.inference_mode():
        out = model(batch["sequence"].to(device))
    output_path = INFERENCE_DIR / f"{sample_id}_inference.npz"
    payload = save_inference_sample(batch, 0, out, output_path, save_full_npz=True)
    for key, value in payload.items():
        if not key.startswith("_"):
            inference_df.loc[row_idx, key] = value

for sample_id in tqdm(full_npz_sample_ids, desc="materialize full inference NPZ"):
    materialize_full_inference_npz(sample_id)

inference_df.to_csv(MANIFEST_DIR / "inference_manifest.csv", index=False)
representative_df = inference_df[inference_df["sample_id"].isin(representative_sample_ids)].copy()
representative_df["selection_rank"] = representative_df["sample_id"].map({sid: i for i, sid in enumerate(representative_sample_ids)})
representative_df = representative_df.sort_values("selection_rank")
representative_df.to_csv(MANIFEST_DIR / "representative_samples.csv", index=False)
print("Visualization samples:", representative_sample_ids)
display(representative_df)

full_npz_df = inference_df[inference_df["sample_id"].isin(full_npz_sample_ids)].copy()
full_npz_df["storage_rank"] = full_npz_df["sample_id"].map({sid: i for i, sid in enumerate(full_npz_sample_ids)})
full_npz_df = full_npz_df.sort_values("storage_rank")
full_npz_df.to_csv(MANIFEST_DIR / "full_npz_samples.csv", index=False)
print("Full-NPZ samples:", full_npz_sample_ids)
display(full_npz_df.head())

storage_plan_df = pd.DataFrame({
    "sample_id": inference_df["sample_id"].astype(str),
    "save_full_npz_arrays": [sid in full_npz_sample_id_set or bool(CONFIG["save_full_npz_for_all_samples"]) for sid in inference_df["sample_id"].astype(str)],
    "save_visualizations": [sid in visualization_sample_id_set for sid in inference_df["sample_id"].astype(str)],
    "selection_phase": "final_after_inference",
})
storage_plan_df.to_csv(MANIFEST_DIR / "sample_storage_plan.csv", index=False)


## PART 2 - RAFT-Large Optical Flow


In [ ]:
weights = Raft_Large_Weights.DEFAULT
raft_transforms = weights.transforms()
raft_model = raft_large(weights=weights, progress=True).to(device).eval()
flow_metadata = {
    "torchvision_version": torchvision.__version__,
    "raft_weight_identifier": str(weights),
    "raft_weight_meta": getattr(weights, "meta", {}),
    "flow_direction_convention": "flow_forward_uv[t] maps input frame t toward input frame t+1; flow_backward_uv[t] maps input frame t+1 toward frame t; uv are dx,dy in resized ConvLSTM input pixels.",
}
with (MANIFEST_DIR / "raft_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(flow_metadata, f, indent=2, default=str)
print(json.dumps({k: flow_metadata[k] for k in ["torchvision_version", "raft_weight_identifier", "flow_direction_convention"]}, indent=2))


In [ ]:
def raft_padded_hw(height: int, width: int) -> tuple[int, int]:
    """RAFT-Large needs inputs at least 128x128 and divisible by 8; analysis is cropped back afterward."""
    min_size = int(CONFIG.get("raft_min_input_size", 128))
    padded_h = int(np.ceil(max(height, min_size) / 8.0) * 8)
    padded_w = int(np.ceil(max(width, min_size) / 8.0) * 8)
    return padded_h, padded_w


def pad_raft_images(images: torch.Tensor, original_hw: tuple[int, int]) -> tuple[torch.Tensor, dict[str, int]]:
    h, w = original_hw
    padded_h, padded_w = raft_padded_hw(h, w)
    pad_bottom = padded_h - h
    pad_right = padded_w - w
    if pad_bottom == 0 and pad_right == 0:
        return images, {"original_h": h, "original_w": w, "padded_h": padded_h, "padded_w": padded_w, "pad_bottom": 0, "pad_right": 0}
    mode = str(CONFIG.get("raft_pad_mode", "replicate"))
    padded = F.pad(images, (0, pad_right, 0, pad_bottom), mode=mode)
    return padded, {"original_h": h, "original_w": w, "padded_h": padded_h, "padded_w": padded_w, "pad_bottom": pad_bottom, "pad_right": pad_right}


def crop_raft_flow(flow: torch.Tensor, pad_info: dict[str, int]) -> torch.Tensor:
    return flow[..., : int(pad_info["original_h"]), : int(pad_info["original_w"])]


def frames_to_raft_pairs(frames: np.ndarray) -> tuple[torch.Tensor, torch.Tensor, dict[str, int]]:
    """Convert [T,H,W] grayscale [0,1] frames to padded RAFT tensors, then crop flow back later."""
    assert frames.ndim == 3, frames.shape
    original_hw = (int(frames.shape[1]), int(frames.shape[2]))
    rgb = np.repeat(frames[:, None], 3, axis=1).astype(np.float32)
    images = torch.from_numpy(rgb)
    images, pad_info = pad_raft_images(images, original_hw)
    img1 = images[:-1]
    img2 = images[1:]
    img1, img2 = raft_transforms(img1, img2)
    return img1, img2, pad_info


def warp_uv(flow_to_warp: torch.Tensor, reference_flow: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample flow_to_warp at coordinates displaced by reference_flow."""
    n, _, h, w = reference_flow.shape
    yy, xx = torch.meshgrid(
        torch.arange(h, device=reference_flow.device),
        torch.arange(w, device=reference_flow.device),
        indexing="ij",
    )
    x = xx[None].float() + reference_flow[:, 0]
    y = yy[None].float() + reference_flow[:, 1]
    valid = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    grid_x = 2.0 * x / max(w - 1, 1) - 1.0
    grid_y = 2.0 * y / max(h - 1, 1) - 1.0
    grid = torch.stack([grid_x, grid_y], dim=-1)
    warped = F.grid_sample(flow_to_warp, grid, mode="bilinear", padding_mode="zeros", align_corners=True)
    return warped, valid


@torch.inference_mode()
def compute_raft_flows(frames: np.ndarray, batch_size: int = 8) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    img1, img2, pad_info = frames_to_raft_pairs(frames)
    forward_chunks = []
    backward_chunks = []
    for start in range(0, img1.shape[0], batch_size):
        end = min(start + batch_size, img1.shape[0])
        a = img1[start:end].to(device)
        b = img2[start:end].to(device)
        forward_chunks.append(crop_raft_flow(raft_model(a, b)[-1], pad_info).detach().cpu())
        backward_chunks.append(crop_raft_flow(raft_model(b, a)[-1], pad_info).detach().cpu())
    forward = torch.cat(forward_chunks, dim=0).numpy().astype(np.float32)
    backward = torch.cat(backward_chunks, dim=0).numpy().astype(np.float32)
    assert forward.shape[-2:] == tuple(frames.shape[-2:]), (forward.shape, frames.shape)
    assert backward.shape[-2:] == tuple(frames.shape[-2:]), (backward.shape, frames.shape)
    return forward, backward, pad_info


def boundary_mask(mask: np.ndarray, width: int = 3) -> np.ndarray:
    mask_u8 = (mask > 0).astype(np.uint8)
    kernel = np.ones((max(1, width), max(1, width)), np.uint8)
    dilated = cv2.dilate(mask_u8, kernel, iterations=1)
    eroded = cv2.erode(mask_u8, kernel, iterations=1)
    return ((dilated - eroded) > 0)


def summarize_region(values: np.ndarray, mask: np.ndarray) -> dict[str, float]:
    if values.ndim != 3:
        raise AssertionError(values.shape)
    rows = []
    for t in range(values.shape[0]):
        region = values[t][mask]
        rows.append(float(region.mean()) if region.size else float("nan"))
    return {"per_transition": rows, "mean": float(np.nanmean(rows)) if len(rows) else float("nan")}


def load_flow_inputs(row: Any) -> dict[str, Any]:
    inference_npz_path = str(getattr(row, "inference_npz_path", ""))
    if inference_npz_path:
        with np.load(inference_npz_path, allow_pickle=False) as data:
            return {
                "frames": data["input_frames"].astype(np.float32),
                "frame_indices": data["sampled_frame_indices"].astype(np.int32),
                "transition_cavity_roi": data["transition_cavity_roi"].astype(bool),
                "transition_boundary_roi": data["transition_boundary_roi"].astype(bool),
                "dynamic_frame_roi_mask": data["dynamic_frame_roi_mask"].astype(np.uint8),
                "dynamic_frame_roi_source": data["dynamic_frame_roi_source"],
                "dynamic_frame_has_ground_truth": data["dynamic_frame_has_ground_truth"].astype(np.uint8),
                "motion_head": data["motion_head_flow_uv"].astype(np.float32),
                "sample_id": str(data["sample_id"]),
                "video_id": str(data["video_id"]),
                "target_idx": int(data["target_idx"]),
            }
    sample_id = str(row.sample_id)
    full_index = sample_id_to_full_index[sample_id]
    sample = test_dataset[full_index]
    frames = sample["sequence"].detach().cpu().numpy().astype(np.float32)[:, 0]
    frame_indices = sample["frame_indices"].detach().cpu().numpy().astype(np.int32)
    video_id = str(sample["video_id"])
    target_idx = int(sample["target_idx"])
    dynamic = generate_dynamic_frame_masks(video_id, frame_indices)
    transition_cavity_roi, transition_boundary_roi = transition_rois_from_frame_masks(
        dynamic["dynamic_frame_roi_mask"],
        boundary_width_px=int(CONFIG["boundary_width_px"]),
    )
    return {
        "frames": frames,
        "frame_indices": frame_indices,
        "transition_cavity_roi": transition_cavity_roi.astype(bool),
        "transition_boundary_roi": transition_boundary_roi.astype(bool),
        "dynamic_frame_roi_mask": dynamic["dynamic_frame_roi_mask"].astype(np.uint8),
        "dynamic_frame_roi_source": dynamic["dynamic_frame_roi_source"],
        "dynamic_frame_has_ground_truth": dynamic["dynamic_frame_has_ground_truth"].astype(np.uint8),
        "motion_head": None,
        "sample_id": sample_id,
        "video_id": video_id,
        "target_idx": target_idx,
    }


def maybe_cast_flow_array(array: np.ndarray) -> np.ndarray:
    if str(CONFIG.get("flow_array_storage_dtype", "float32")).lower() == "float16":
        return array.astype(np.float16)
    return array.astype(np.float32)


def compute_flow_payload(row: Any, output_path: str | Path, save_full_npz: bool) -> dict[str, Any]:
    payload = load_flow_inputs(row)
    frames = payload["frames"]
    frame_indices = payload["frame_indices"]
    transition_cavity_roi = payload["transition_cavity_roi"]
    transition_boundary_roi = payload["transition_boundary_roi"]
    dynamic_frame_roi_mask = payload["dynamic_frame_roi_mask"]
    dynamic_frame_roi_source = payload["dynamic_frame_roi_source"]
    dynamic_frame_has_ground_truth = payload["dynamic_frame_has_ground_truth"]
    motion_head = payload["motion_head"]
    sample_id = payload["sample_id"]
    video_id = payload["video_id"]
    target_idx = payload["target_idx"]
    forward, backward, raft_pad_info = compute_raft_flows(frames, batch_size=int(CONFIG["raft_transition_batch_size"]))
    fwd_t = torch.from_numpy(forward).to(device)
    bwd_t = torch.from_numpy(backward).to(device)
    bwd_warped, valid = warp_uv(bwd_t, fwd_t)
    fb_error = torch.linalg.vector_norm(fwd_t + bwd_warped, dim=1).detach().cpu().numpy().astype(np.float32)
    valid_mask = (valid.detach().cpu().numpy() & (fb_error <= float(CONFIG["flow_valid_error_threshold_px"]))).astype(np.uint8)
    mag = np.linalg.norm(forward, axis=1).astype(np.float32)
    angle = np.arctan2(forward[:, 1], forward[:, 0]).astype(np.float32)
    assert transition_cavity_roi.shape == mag.shape, (transition_cavity_roi.shape, mag.shape)
    assert transition_boundary_roi.shape == mag.shape, (transition_boundary_roi.shape, mag.shape)
    whole_motion = np.array([float(mag[t].mean()) for t in range(mag.shape[0])], dtype=np.float32)
    lv_motion = np.array([float(np.nanmean(mag[t][transition_cavity_roi[t]])) if transition_cavity_roi[t].any() else float("nan") for t in range(mag.shape[0])], dtype=np.float32)
    lv_boundary_motion = np.array([float(np.nanmean(mag[t][transition_boundary_roi[t]])) if transition_boundary_roi[t].any() else float("nan") for t in range(mag.shape[0])], dtype=np.float32)
    motion_head_mag = np.linalg.norm(motion_head, axis=1).astype(np.float32) if motion_head is not None else np.full_like(mag, np.nan, dtype=np.float32)
    analysis_path = FLOW_ANALYSIS_DIR / f"{sample_id}_flow_analysis.npz"
    analysis_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        analysis_path,
        flow_forward_magnitude=maybe_cast_flow_array(mag),
        valid_flow_mask=valid_mask,
        transition_cavity_roi=transition_cavity_roi.astype(np.uint8),
        transition_boundary_roi=transition_boundary_roi.astype(np.uint8),
        whole_frame_motion=whole_motion,
        lv_motion=lv_motion,
        lv_boundary_motion=lv_boundary_motion,
        sampled_frame_indices=frame_indices,
        transition_start_frame_indices=frame_indices[:-1],
        transition_end_frame_indices=frame_indices[1:],
        target_idx=np.array(target_idx, dtype=np.int32),
        sample_id=np.array(sample_id),
        video_id=np.array(video_id),
    )
    flow_npz_path = ""
    if save_full_npz:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            output_path,
            flow_forward_uv=maybe_cast_flow_array(forward),
            flow_backward_uv=maybe_cast_flow_array(backward),
            flow_forward_magnitude=maybe_cast_flow_array(mag),
            flow_forward_angle=maybe_cast_flow_array(angle),
            forward_backward_error=maybe_cast_flow_array(fb_error),
            valid_flow_mask=valid_mask,
            dynamic_frame_roi_mask=dynamic_frame_roi_mask,
            dynamic_frame_roi_source=dynamic_frame_roi_source,
            dynamic_frame_has_ground_truth=dynamic_frame_has_ground_truth,
            transition_cavity_roi=transition_cavity_roi.astype(np.uint8),
            transition_boundary_roi=transition_boundary_roi.astype(np.uint8),
            motion_head_flow_uv=maybe_cast_flow_array(motion_head) if motion_head is not None else np.empty((0,), dtype=np.float16),
            motion_head_flow_magnitude=maybe_cast_flow_array(motion_head_mag) if motion_head is not None else np.empty((0,), dtype=np.float16),
            whole_frame_motion=whole_motion,
            lv_motion=lv_motion,
            lv_boundary_motion=lv_boundary_motion,
            sampled_frame_indices=frame_indices,
            transition_start_frame_indices=frame_indices[:-1],
            transition_end_frame_indices=frame_indices[1:],
            target_idx=np.array(target_idx, dtype=np.int32),
            sample_id=np.array(sample_id),
            video_id=np.array(video_id),
            metadata_json=np.array(json.dumps({
                "sample_id": sample_id,
                "video_id": video_id,
                "target_idx": target_idx,
                "flow_metadata": flow_metadata,
                "raft_padding": raft_pad_info,
                "flow_array_storage_dtype": str(CONFIG.get("flow_array_storage_dtype", "float32")),
                "dynamic_roi_note": "Transition cavity ROI is the union of dynamic masks for frames t and t+1. Transition boundary ROI is the union of each frame boundary ring. GT masks replace predicted masks where available.",
            }, default=str)),
        )
        flow_npz_path = str(output_path)
    transition_rows = []
    for t in range(mag.shape[0]):
        transition_rows.append({
            "sample_id": sample_id,
            "video_id": video_id,
            "transition_idx": int(t),
            "start_frame_idx": int(frame_indices[t]),
            "end_frame_idx": int(frame_indices[t + 1]),
            "whole_frame_motion": float(whole_motion[t]),
            "lv_cavity_motion": float(lv_motion[t]),
            "lv_boundary_motion": float(lv_boundary_motion[t]),
            "forward_backward_error_mean": float(np.nanmean(fb_error[t])),
            "valid_flow_fraction": float(valid_mask[t].mean()),
            "transition_cavity_roi_area_fraction": float(transition_cavity_roi[t].mean()),
            "transition_boundary_roi_area_fraction": float(transition_boundary_roi[t].mean()),
            "save_full_npz_arrays": bool(save_full_npz),
        })
    return {
        "sample_id": sample_id,
        "video_id": video_id,
        "save_full_npz_arrays": bool(save_full_npz),
        "flow_npz_path": flow_npz_path,
        "flow_analysis_npz_path": str(analysis_path),
        "whole_frame_motion_mean": float(np.mean(whole_motion)),
        "lv_motion_mean": float(np.nanmean(lv_motion)),
        "lv_boundary_motion_mean": float(np.nanmean(lv_boundary_motion)),
        "forward_backward_error_mean": float(np.nanmean(fb_error)),
        "valid_flow_fraction": float(valid_mask.mean()),
        "raft_original_h": int(raft_pad_info["original_h"]),
        "raft_original_w": int(raft_pad_info["original_w"]),
        "raft_padded_h": int(raft_pad_info["padded_h"]),
        "raft_padded_w": int(raft_pad_info["padded_w"]),
        "gt_roi_frame_count": int(dynamic_frame_has_ground_truth.sum()),
        "transition_cavity_roi_mean_area": float(transition_cavity_roi.mean()),
        "transition_boundary_roi_mean_area": float(transition_boundary_roi.mean()),
        "_transition_rows": transition_rows,
    }

flow_rows = []
flow_transition_rows = []
for row in tqdm(inference_df.itertuples(index=False), total=len(inference_df), desc="RAFT flow"):
    sample_id = str(row.sample_id)
    save_full_npz = bool(CONFIG["save_full_npz_for_all_samples"]) or sample_id in full_npz_sample_id_set
    payload = compute_flow_payload(row, FLOW_DIR / f"{sample_id}_raft_flow.npz", save_full_npz=save_full_npz)
    flow_transition_rows.extend(payload.pop("_transition_rows"))
    flow_rows.append(payload)
flow_df = pd.DataFrame(flow_rows)
flow_transition_df = pd.DataFrame(flow_transition_rows)
flow_df.to_csv(MANIFEST_DIR / "flow_manifest.csv", index=False)
flow_transition_df.to_csv(MANIFEST_DIR / "flow_transition_metrics.csv", index=False)
display(flow_df.head())
display(flow_transition_df.head())


## Frame and Grad-CAM Alignment Checks


In [ ]:

def gradcam_npz_path(sample_id: str, cam_type: str) -> Path:
    return GRADCAM_MODEL_DIR / cam_type / "npz" / f"{sample_id}_{cam_type}_gradcam.npz"


def transition_saliency(cams: np.ndarray) -> np.ndarray:
    assert cams.ndim == 3, cams.shape
    return ((cams[:-1] + cams[1:]) / 2.0).astype(np.float32)


def transition_saliency_path(sample_id: str, cam_type: str) -> Path:
    return RUN_DIR / "transition_saliency" / cam_type / f"{sample_id}_{cam_type}_transition_saliency.npz"


def load_transition_saliency(sample_id: str, cam_type: str, cam_key: str) -> tuple[np.ndarray, dict[str, Any]]:
    path = transition_saliency_path(sample_id, cam_type)
    with np.load(path, allow_pickle=False) as data:
        sal = data["transition_saliency"].astype(np.float32)
        metadata = json.loads(str(data["metadata_json"])) if "metadata_json" in data.files else {}
    assert sal.shape[0] == int(CONFIG["sequence_length"]) - 1, sal.shape
    return sal, metadata


def pearson_corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.std(a) <= 1e-12 or np.std(b) <= 1e-12:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def spearman_corr(a: np.ndarray, b: np.ndarray) -> float:
    ar = pd.Series(np.asarray(a).ravel()).rank(method="average").to_numpy()
    br = pd.Series(np.asarray(b).ravel()).rank(method="average").to_numpy()
    return pearson_corr(ar, br)


def topk_mask(x: np.ndarray, fraction: float) -> np.ndarray:
    flat = np.asarray(x, dtype=np.float32).ravel()
    k = max(1, int(round(float(fraction) * flat.size)))
    threshold = np.partition(flat, flat.size - k)[flat.size - k]
    return x >= threshold


def binary_iou_dice(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    inter = int(np.logical_and(a, b).sum())
    union = int(np.logical_or(a, b).sum())
    denom = int(a.sum() + b.sum())
    iou = float(inter / union) if union else float("nan")
    dice = float(2 * inter / denom) if denom else float("nan")
    return iou, dice


def gradcam_batch_for_sample(sample_id: str) -> dict[str, Any]:
    full_index = sample_id_to_full_index[str(sample_id)]
    loader = DataLoader(Subset(test_dataset, [full_index]), batch_size=1, shuffle=False, num_workers=0)
    batch = next(iter(loader))
    batch["sequence"] = batch["sequence"].to(device)
    return batch


def run_gradcam_for_sample(sample_id: str, cam_type: str) -> tuple[Any, dict[str, Any], Path]:
    batch = gradcam_batch_for_sample(sample_id)
    sequence = batch["sequence"]
    if cam_type == "encoder_bottleneck":
        result = encoder_bottleneck_ef_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std)
        target_layer = "model.bottleneck_encoder"
    elif cam_type == "temporal_representation":
        result = temporal_representation_ef_probe_gradcam(model, sequence, ef_mean=ef_mean, ef_std=ef_std)
        target_layer = "model.temporal_features + ef_pool/ef_head probes"
    else:
        raise ValueError(f"Unsupported CAM type: {cam_type}")
    pred_ref = float(inference_df.loc[inference_df["sample_id"] == sample_id, "ef_pred"].iloc[0])
    assert abs(float(result.pred_ef) - pred_ref) < 1e-3, (sample_id, cam_type, result.pred_ef, pred_ref)
    diagnostics_path = GRADCAM_MODEL_DIR / cam_type / "diagnostics" / f"{sample_id}_{cam_type}_diagnostics.csv"
    diagnostics_path.parent.mkdir(parents=True, exist_ok=True)
    result.temporal_diagnostics.to_csv(diagnostics_path, index=False)
    return result, batch, diagnostics_path


def spatial_temporal_metrics_from_arrays(sample_id: str, cam_type: str, sal: np.ndarray, flow_analysis_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    with np.load(flow_analysis_path, allow_pickle=False) as flow:
        mag = flow["flow_forward_magnitude"].astype(np.float32)
        transition_cavity_roi = flow["transition_cavity_roi"].astype(bool)
        transition_boundary_roi = flow["transition_boundary_roi"].astype(bool)
        valid = flow["valid_flow_mask"].astype(bool)
    assert sal.shape == mag.shape, (sample_id, cam_type, sal.shape, mag.shape)
    rows = []
    temporal_rows = []
    for t in range(sal.shape[0]):
        masks = {
            "whole_frame": valid[t],
            "lv_cavity_dynamic_union": valid[t] & transition_cavity_roi[t],
            "lv_boundary_dynamic_union": valid[t] & transition_boundary_roi[t],
        }
        for region_name, region_mask in masks.items():
            if not region_mask.any():
                continue
            sal_region = sal[t][region_mask]
            mag_region = mag[t][region_mask]
            sal_top = topk_mask(sal_region, float(CONFIG["topk_fraction"]))
            mag_top = topk_mask(mag_region, float(CONFIG["topk_fraction"]))
            iou, dice = binary_iou_dice(sal_top, mag_top)
            rows.append({
                "sample_id": sample_id,
                "cam_type": cam_type,
                "transition_idx": t,
                "region": region_name,
                "pearson": pearson_corr(sal_region, mag_region),
                "spearman": spearman_corr(sal_region, mag_region),
                "topk_iou": iou,
                "topk_dice": dice,
                "motion_weighted_saliency": float(np.sum(sal_region * mag_region) / max(float(np.sum(mag_region)), 1e-8)),
                "mean_saliency": float(np.mean(sal_region)),
                "mean_flow_magnitude": float(np.mean(mag_region)),
            })
        temporal_rows.append({
            "sample_id": sample_id,
            "cam_type": cam_type,
            "transition_idx": t,
            "mean_lv_flow_magnitude": float(np.nanmean(mag[t][transition_cavity_roi[t]])) if transition_cavity_roi[t].any() else float("nan"),
            "mean_transition_saliency": float(np.nanmean(sal[t][transition_cavity_roi[t]])) if transition_cavity_roi[t].any() else float("nan"),
            "whole_frame_flow_magnitude": float(np.nanmean(mag[t])),
            "whole_frame_transition_saliency": float(np.nanmean(sal[t])),
        })
    return pd.DataFrame(rows), pd.DataFrame(temporal_rows)

gradcam_manifest_rows = []
transition_rows = []
metric_tables = []
temporal_tables = []
for row in tqdm(flow_df.itertuples(index=False), total=len(flow_df), desc="generate Grad-CAM and compare flow"):
    sample_id = str(row.sample_id)
    save_full_npz = bool(CONFIG["save_full_npz_for_all_samples"]) or sample_id in full_npz_sample_id_set
    flow_analysis_path = Path(row.flow_analysis_npz_path)
    assert flow_analysis_path.exists(), flow_analysis_path
    for cam_type in CONFIG["gradcam_layers"]:
        result, batch, diagnostics_path = run_gradcam_for_sample(sample_id, cam_type)
        cam_npz = gradcam_npz_path(sample_id, cam_type)
        if save_full_npz:
            save_gradcam_npz(
                cam_npz,
                result,
                batch,
                model_name="ef_primary_motion",
                checkpoint_metadata=checkpoint_metadata,
                cam_type=cam_type,
                target_layer="model.bottleneck_encoder" if cam_type == "encoder_bottleneck" else "model.temporal_features + ef_pool/ef_head probes",
                dataset_metrics=dataset_metrics,
                diagnostics_csv_path=diagnostics_path,
            )
        sal = transition_saliency(getattr(result, str(CONFIG["gradcam_analysis_cam_key"]))).astype(np.float32)
        sal_path = transition_saliency_path(sample_id, cam_type)
        if save_full_npz:
            sal_path.parent.mkdir(parents=True, exist_ok=True)
            sal_to_save = sal.astype(np.float16) if str(CONFIG.get("flow_array_storage_dtype")) == "float16" else sal
            np.savez_compressed(
                sal_path,
                transition_saliency=sal_to_save,
                sample_id=np.array(sample_id),
                cam_type=np.array(cam_type),
                cam_key=np.array(str(CONFIG["gradcam_analysis_cam_key"])),
                metadata_json=np.array(json.dumps({"sample_id": sample_id, "cam_type": cam_type, "cam_key": CONFIG["gradcam_analysis_cam_key"]}, default=str)),
            )
        metrics, temporal = spatial_temporal_metrics_from_arrays(sample_id, cam_type, sal, flow_analysis_path)
        metric_tables.append(metrics)
        temporal_tables.append(temporal)
        gradcam_manifest_rows.append({
            "sample_id": sample_id,
            "cam_type": cam_type,
            "model_name": "ef_primary_motion",
            "save_full_npz_arrays": bool(save_full_npz),
            "gradcam_npz_path": str(cam_npz) if save_full_npz else "",
            "diagnostics_csv_path": str(diagnostics_path),
            "transition_saliency_npz_path": str(sal_path) if save_full_npz else "",
            "predicted_ef": float(result.pred_ef),
            "reference_predicted_ef": float(inference_df.loc[inference_df["sample_id"] == sample_id, "ef_pred"].iloc[0]),
            "max_abs_signed_cam": float(np.max(np.abs(result.signed_raw_cams))),
            "positive_cam_max": float(np.max(result.positive_cams)),
            "activation_shape": str(result.activation_shape),
            "gradient_shape": str(result.gradient_shape),
            "temporal_features_shape": str(result.temporal_features_shape),
        })
        transition_rows.append({
            "sample_id": sample_id,
            "cam_type": cam_type,
            "save_full_npz_arrays": bool(save_full_npz),
            "transition_saliency_npz_path": str(sal_path) if save_full_npz else "",
            "transition_count": int(sal.shape[0]),
            "cam_key": str(CONFIG["gradcam_analysis_cam_key"]),
        })

gradcam_manifest_df = pd.DataFrame(gradcam_manifest_rows)
transition_df = pd.DataFrame(transition_rows)
metrics_df = pd.concat(metric_tables, ignore_index=True) if metric_tables else pd.DataFrame()
temporal_df = pd.concat(temporal_tables, ignore_index=True) if temporal_tables else pd.DataFrame()
summary_metrics = metrics_df.groupby(["cam_type", "region"], dropna=False).agg({
    "pearson": "mean",
    "spearman": "mean",
    "topk_iou": "mean",
    "topk_dice": "mean",
    "motion_weighted_saliency": "mean",
}).reset_index() if not metrics_df.empty else pd.DataFrame()

gradcam_manifest_df.to_csv(MANIFEST_DIR / "generated_gradcam_manifest.csv", index=False)
transition_df.to_csv(MANIFEST_DIR / "transition_saliency_manifest.csv", index=False)
metrics_df.to_csv(MANIFEST_DIR / "motion_saliency_spatial_metrics.csv", index=False)
temporal_df.to_csv(MANIFEST_DIR / "motion_saliency_temporal_curves.csv", index=False)
summary_metrics.to_csv(MANIFEST_DIR / "motion_saliency_summary_metrics.csv", index=False)
display(gradcam_manifest_df.head())
display(summary_metrics)


## PART 3 - Visualization Utilities


In [ ]:
def normalize01(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    lo, hi = float(np.nanmin(x)), float(np.nanmax(x))
    if hi - lo <= eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - lo) / (hi - lo)).astype(np.float32)


def overlay_heat(frame: np.ndarray, heat: np.ndarray, cmap_name: str = "turbo", alpha: float = 0.45) -> np.ndarray:
    frame = np.clip(frame, 0, 1)
    heat = np.clip(heat, 0, 1)
    rgb = np.repeat(frame[..., None], 3, axis=2)
    color = plt.get_cmap(cmap_name)(heat)[..., :3]
    local_alpha = alpha * heat[..., None]
    return np.clip((1 - local_alpha) * rgb + local_alpha * color, 0, 1)


def flow_to_hsv(flow_uv: np.ndarray, magnitude: np.ndarray | None = None) -> np.ndarray:
    if magnitude is None:
        magnitude = np.linalg.norm(flow_uv, axis=0)
    angle = np.arctan2(flow_uv[1], flow_uv[0])
    hue = ((angle + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    sat = np.full_like(hue, 255, dtype=np.uint8)
    val = (normalize01(magnitude) * 255).astype(np.uint8)
    hsv = np.stack([hue, sat, val], axis=-1)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB).astype(np.float32) / 255.0


def draw_sparse_flow(ax, flow_uv: np.ndarray, step: int = 12, color: str = "cyan") -> None:
    h, w = flow_uv.shape[1:]
    yy, xx = np.mgrid[step // 2:h:step, step // 2:w:step]
    u = flow_uv[0, yy, xx]
    v = flow_uv[1, yy, xx]
    ax.quiver(xx, yy, u, v, color=color, angles="xy", scale_units="xy", scale=1.0, width=0.003)


def save_transition_video(frames: list[np.ndarray], output_path: Path, fps: int = 5) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    first = (np.clip(frames[0], 0, 1) * 255).astype(np.uint8)
    h, w = first.shape[:2]
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), float(fps), (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer: {output_path}")
    try:
        for frame in frames:
            image = (np.clip(frame, 0, 1) * 255).astype(np.uint8)
            writer.write(cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    finally:
        writer.release()


## PART 4 - Transition-Level Grad-CAM


In [ ]:

display(transition_df.head())
display(gradcam_manifest_df.head())


## PART 5 - Motion vs Saliency Metrics


In [ ]:

display(summary_metrics)
display(metrics_df.head())
display(temporal_df.head())


## PART 3 and 5 - Representative Spatial Figures and Videos


In [ ]:
def make_sample_visualizations(sample_id: str, cam_type: str) -> list[dict[str, Any]]:
    inf_path = Path(inference_df[inference_df["sample_id"] == sample_id].iloc[0]["inference_npz_path"])
    flow_path = Path(flow_df[flow_df["sample_id"] == sample_id].iloc[0]["flow_npz_path"])
    sal_analysis, _ = load_transition_saliency(sample_id, cam_type, CONFIG["gradcam_analysis_cam_key"])
    sal_visual, _ = load_transition_saliency(sample_id, cam_type, CONFIG["gradcam_visual_cam_key"])
    with np.load(inf_path, allow_pickle=False) as inf, np.load(flow_path, allow_pickle=False) as flow:
        frames = inf["input_frames"].astype(np.float32)
        frame_indices = inf["sampled_frame_indices"].astype(np.int32)
        transition_cavity_roi = flow["transition_cavity_roi"].astype(bool)
        transition_boundary_roi = flow["transition_boundary_roi"].astype(bool)
        mag = flow["flow_forward_magnitude"].astype(np.float32)
        uv = flow["flow_forward_uv"].astype(np.float32)
        angle = flow["flow_forward_angle"].astype(np.float32)
    rows = []
    transitions = mag.shape[0]
    cols = 6
    selected_t = np.linspace(0, transitions - 1, min(8, transitions)).round().astype(int)
    for figure_kind in ["flow_magnitude_overlay", "sparse_flow_vectors", "lv_restricted_flow", "flow_direction", "gradcam_vs_flow_side_by_side", "gradcam_plus_flow_overlay"]:
        fig, axes = plt.subplots(len(selected_t), 2 if figure_kind == "gradcam_vs_flow_side_by_side" else 1, figsize=(6 if figure_kind == "gradcam_vs_flow_side_by_side" else 3, 2.6 * len(selected_t)))
        axes_arr = np.asarray(axes).reshape(len(selected_t), -1)
        for row_idx, t in enumerate(selected_t):
            frame = frames[t]
            flow_heat = normalize01(mag[t])
            sal_heat = normalize01(sal_visual[t])
            if figure_kind == "flow_magnitude_overlay":
                image = overlay_heat(frame, flow_heat, cmap_name="magma", alpha=0.55)
                axes_arr[row_idx, 0].imshow(image)
            elif figure_kind == "sparse_flow_vectors":
                axes_arr[row_idx, 0].imshow(frame, cmap="gray", vmin=0, vmax=1)
                draw_sparse_flow(axes_arr[row_idx, 0], uv[t], step=12)
            elif figure_kind == "lv_restricted_flow":
                masked = np.where(transition_cavity_roi[t], flow_heat, 0.0)
                axes_arr[row_idx, 0].imshow(overlay_heat(frame, masked, cmap_name="magma", alpha=0.65))
                axes_arr[row_idx, 0].contour(transition_cavity_roi[t], levels=[0.5], colors="cyan", linewidths=0.8)
                axes_arr[row_idx, 0].contour(transition_boundary_roi[t], levels=[0.5], colors="white", linewidths=0.6)
            elif figure_kind == "flow_direction":
                axes_arr[row_idx, 0].imshow(flow_to_hsv(uv[t], mag[t]))
            elif figure_kind == "gradcam_vs_flow_side_by_side":
                axes_arr[row_idx, 0].imshow(overlay_heat(frame, sal_heat, cmap_name="turbo", alpha=0.55))
                axes_arr[row_idx, 0].set_title("Grad-CAM transition saliency", fontsize=8)
                axes_arr[row_idx, 1].imshow(overlay_heat(frame, flow_heat, cmap_name="magma", alpha=0.55))
                axes_arr[row_idx, 1].set_title("RAFT flow magnitude", fontsize=8)
            elif figure_kind == "gradcam_plus_flow_overlay":
                image = overlay_heat(frame, sal_heat, cmap_name="turbo", alpha=0.45)
                axes_arr[row_idx, 0].imshow(image)
                draw_sparse_flow(axes_arr[row_idx, 0], uv[t], step=14, color="white")
            for ax in axes_arr[row_idx]:
                ax.axis("off")
                ax.set_title(f"t{t:02d}: src {int(frame_indices[t])}->{int(frame_indices[t+1])}", fontsize=8)
        fig.suptitle(f"{sample_id} | {cam_type} | {figure_kind} | dynamic transition ROI", fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        out_path = FIGURE_DIR / cam_type / figure_kind / f"{sample_id}_{cam_type}_{figure_kind}.png"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        rows.append({"sample_id": sample_id, "cam_type": cam_type, "figure_kind": figure_kind, "figure_path": str(out_path)})
    # Videos: flow magnitude and Grad-CAM+flow overlay.
    video_frames_flow = [overlay_heat(frames[t], normalize01(mag[t]), cmap_name="magma", alpha=0.55) for t in range(transitions)]
    video_path_flow = VIDEO_DIR / cam_type / "flow_magnitude_overlay" / f"{sample_id}_{cam_type}_flow_magnitude_overlay.mp4"
    save_transition_video(video_frames_flow, video_path_flow, fps=int(CONFIG["video_fps"]))
    rows.append({"sample_id": sample_id, "cam_type": cam_type, "figure_kind": "flow_magnitude_overlay_video", "figure_path": str(video_path_flow)})
    video_frames_combo = []
    for t in range(transitions):
        canvas = overlay_heat(frames[t], normalize01(sal_visual[t]), cmap_name="turbo", alpha=0.45)
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.imshow(canvas)
        draw_sparse_flow(ax, uv[t], step=14, color="white")
        ax.axis("off")
        fig.canvas.draw()
        image = np.asarray(fig.canvas.buffer_rgba())[..., :3].astype(np.float32) / 255.0
        plt.close(fig)
        video_frames_combo.append(image)
    video_path_combo = VIDEO_DIR / cam_type / "gradcam_plus_flow_overlay" / f"{sample_id}_{cam_type}_gradcam_plus_flow_overlay.mp4"
    save_transition_video(video_frames_combo, video_path_combo, fps=int(CONFIG["video_fps"]))
    rows.append({"sample_id": sample_id, "cam_type": cam_type, "figure_kind": "gradcam_plus_flow_overlay_video", "figure_path": str(video_path_combo)})
    return rows

visual_rows = []
for sample_id in tqdm(representative_sample_ids, desc="representative visualizations"):
    for cam_type in CONFIG["gradcam_layers"]:
        visual_rows.extend(make_sample_visualizations(sample_id, cam_type))
visual_df = pd.DataFrame(visual_rows)
visual_df.to_csv(MANIFEST_DIR / "visualization_manifest.csv", index=False)
display(visual_df.head())


## PART 6 - Temporal Analysis


In [ ]:
temporal_summary_rows = []
for (sample_id, cam_type), group in temporal_df.groupby(["sample_id", "cam_type"]):
    temporal_summary_rows.append({
        "sample_id": sample_id,
        "cam_type": cam_type,
        "lv_temporal_pearson": pearson_corr(group["mean_transition_saliency"].to_numpy(), group["mean_lv_flow_magnitude"].to_numpy()),
        "lv_temporal_spearman": spearman_corr(group["mean_transition_saliency"].to_numpy(), group["mean_lv_flow_magnitude"].to_numpy()),
        "whole_frame_temporal_pearson": pearson_corr(group["whole_frame_transition_saliency"].to_numpy(), group["whole_frame_flow_magnitude"].to_numpy()),
        "whole_frame_temporal_spearman": spearman_corr(group["whole_frame_transition_saliency"].to_numpy(), group["whole_frame_flow_magnitude"].to_numpy()),
    })
temporal_summary_df = pd.DataFrame(temporal_summary_rows)
temporal_summary_df.to_csv(MANIFEST_DIR / "motion_saliency_temporal_correlations.csv", index=False)

plot_rows = []
for (sample_id, cam_type), group in temporal_df.groupby(["sample_id", "cam_type"]):
    fig, ax1 = plt.subplots(figsize=(8, 3.2))
    x = group["transition_idx"].to_numpy()
    ax1.plot(x, group["mean_lv_flow_magnitude"], color="tab:blue", label="mean LV flow magnitude")
    ax1.set_xlabel("transition index")
    ax1.set_ylabel("LV flow magnitude", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(x, group["mean_transition_saliency"], color="tab:red", label="transition saliency")
    ax2.set_ylabel("transition saliency", color="tab:red")
    target = int(CONFIG["target_idx"])
    ax1.axvline(target - 1, color="black", linestyle="--", linewidth=0.8, label="target-adjacent transition")
    fig.suptitle(f"{sample_id} | {cam_type} | LV motion vs Grad-CAM transition saliency")
    fig.tight_layout()
    path = FIGURE_DIR / cam_type / "temporal_curves" / f"{sample_id}_{cam_type}_temporal_motion_saliency_curves.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    plot_rows.append({"sample_id": sample_id, "cam_type": cam_type, "temporal_curve_path": str(path)})
plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(MANIFEST_DIR / "temporal_curve_manifest.csv", index=False)
display(temporal_summary_df)


## Required Outputs and Summary


In [ ]:
expected_outputs = [
    RUN_DIR / "config.json",
    MANIFEST_DIR / "checkpoint_metadata.json",
    MANIFEST_DIR / "raft_metadata.json",
    MANIFEST_DIR / "inference_manifest.csv",
    MANIFEST_DIR / "flow_manifest.csv",
    MANIFEST_DIR / "flow_transition_metrics.csv",
    MANIFEST_DIR / "representative_samples.csv",
    MANIFEST_DIR / "full_npz_samples.csv",
    MANIFEST_DIR / "sample_storage_plan.csv",
    MANIFEST_DIR / "generated_gradcam_manifest.csv",
    MANIFEST_DIR / "transition_saliency_manifest.csv",
    MANIFEST_DIR / "motion_saliency_spatial_metrics.csv",
    MANIFEST_DIR / "motion_saliency_summary_metrics.csv",
    MANIFEST_DIR / "motion_saliency_temporal_curves.csv",
    MANIFEST_DIR / "motion_saliency_temporal_correlations.csv",
    MANIFEST_DIR / "visualization_manifest.csv",
    MANIFEST_DIR / "temporal_curve_manifest.csv",
]
for path in expected_outputs:
    print(path, "exists=", path.exists())

summary = {
    "run_mode": RUN_MODE,
    "active_samples": int(len(inference_df)),
    "visualization_samples": representative_sample_ids,
    "full_npz_samples": full_npz_sample_ids,
    "save_full_npz_for_all_samples": bool(CONFIG["save_full_npz_for_all_samples"]),
    "flow_array_storage_dtype": str(CONFIG.get("flow_array_storage_dtype", "float32")),
    "gradcam_source_dir": str(GRADCAM_MODEL_DIR),
    "gradcam_generation_note": "Notebook 16 generates EF Grad-CAM directly using the same encoder_bottleneck_ef_gradcam and temporal_representation_ef_probe_gradcam functions used by notebook 14.",
    "model_used": "ef_primary_motion only",
    "raft_weight_identifier": str(weights),
    "torchvision_version": torchvision.__version__,
    "transition_saliency_rule": "S_t = (CAM_t + CAM_(t+1)) / 2, aligned to RAFT flow transition t -> t+1",
    "lv_roi_note": "LV and LV-boundary flow use dynamic transition ROIs. Every sampled frame is segmented by running the segmentation-only bidirectional ConvLSTM with that frame shifted to the center of a 23-frame input; available ground-truth ED/ES masks replace predicted masks; transition cavity and boundary ROIs are unions across frames t and t+1.",
    "output_dir": str(RUN_DIR),
}
with (MANIFEST_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
